In [1]:
import pandas as pd
from tqdm import tqdm
from binance import Client

In [2]:
client = Client()

In [3]:
info = client.get_exchange_info()

In [4]:
symbols = [symbol["symbol"] for symbol in info["symbols"]]
symbols = [symbol for symbol in symbols if symbol.endswith("USDT")]
symbols = [symbol for symbol in symbols if all(x not in symbol for x in ["UP", "DOWN", "BEAR", "BULL"])]

In [5]:
klines = {}
for symbol in tqdm(symbols):
    klines[symbol] = client.get_historical_klines(symbol, "1m", "30 min ago UTC")

100%|██████████| 525/525 [04:37<00:00,  1.89it/s]


In [6]:
cum_returns, cum_symbols = [], []
for symbol in symbols:
    if len(klines[symbol]) > 0:
        cumret = (pd.DataFrame(klines[symbol])[4].astype(float).pct_change() + 1).prod() - 1
        cum_returns.append(cumret)
        cum_symbols.append(symbol)
cum_df = pd.DataFrame(cum_returns, index=pd.Series(cum_symbols), columns=pd.Series(["cum_returns"]))

In [7]:
(cum_df.cum_returns.nlargest(10) * 100).map("{:,.2f}%".format)

PUNDIXUSDT    5.44%
STOUSDT       2.53%
FUNUSDT       2.29%
SIGNUSDT      2.12%
GPSUSDT       0.97%
BMTUSDT       0.90%
DOGSUSDT      0.89%
CETUSUSDT     0.86%
MOVEUSDT      0.78%
BIOUSDT       0.75%
Name: cum_returns, dtype: object

In [8]:
(cum_df.cum_returns.nsmallest(10) * 100).map("{:,.2f}%".format)

ASRUSDT        -5.69%
MUBARAKUSDT    -4.79%
ACMUSDT        -2.10%
PIVXUSDT       -1.63%
FLMUSDT        -1.33%
ATMUSDT        -1.23%
TUTUSDT        -1.19%
BICOUSDT       -1.01%
VIRTUALUSDT    -0.97%
PORTOUSDT      -0.91%
Name: cum_returns, dtype: object